# RFM

In [5]:
import pandas as pd
import numpy as np


In [14]:
# Option 2: Using a proper raw string with single backslashes
df = pd.read_excel(r'C:\Users\HP\OneDrive\Desktop\kpmg\SEM 3\machine learning\RFM\Customer-Transactions.xlsx')

In [16]:
df.head(10)

,Order_ID,Customer_ID,Order_Date,Region,Product,Quantity,Unit_Price,Discount_Pct,Sales,Payment_Mode
0,ORD109513,1001,2025-05-10,West,Tablet Accessories,2,3103.00,10,5585.40,Credit Card
1,ORD102822,1001,2026-07-29,North,Keyboard,2,2055.71,5,3905.85,Credit Card
2,ORD101849,1002,2025-05-22,West,Headphones,1,9164.46,20,7331.57,UPI
3,ORD100170,1002,2025-06-15,East,Smartwatch,1,4012.69,10,3611.42,Net Banking
4,ORD101546,1002,2025-11-21,South,Printer,1,20160.30,5,19152.28,Net Banking
5,ORD105856,1002,2026-02-10,North,Mouse,1,3908.47,5,3713.05,Cash
6,ORD101654,1002,2026-04-19,East,Smartphone,1,24738.40,15,21027.64,Debit Card
7,ORD108372,1003,2025-01-14,East,Monitor,5,11717.78,0,58588.90,Net Banking
8,ORD103994,1003,2025-08-14,South,Laptop,2,103869.64,20,166191.42,Credit Card
9,ORD101321,1003,2026-04-07,North,Smartphone,1,75308.63,20,60246.90,Credit Card


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Order_ID      10000 non-null  str           
 1   Customer_ID   10000 non-null  int64         
 2   Order_Date    10000 non-null  datetime64[us]
 3   Region        10000 non-null  str           
 4   Product       10000 non-null  str           
 5   Quantity      10000 non-null  int64         
 6   Unit_Price    10000 non-null  float64       
 7   Discount_Pct  10000 non-null  int64         
 8   Sales         10000 non-null  float64       
 9   Payment_Mode  10000 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(3), str(4)
memory usage: 1.1 MB


In [21]:
print("DataFrame shape:", df.shape)

print("DataFrame columns:", df.columns)
print("DataFrame data types:\n", df.dtypes)
print("DataFrame missing values:\n", df.isnull().sum())

DataFrame shape: (10000, 10)
DataFrame columns: Index(['Order_ID', 'Customer_ID', 'Order_Date', 'Region', 'Product',
       'Quantity', 'Unit_Price', 'Discount_Pct', 'Sales', 'Payment_Mode'],
      dtype='str')
DataFrame data types:
 Order_ID                   str
Customer_ID              int64
Order_Date      datetime64[us]
Region                     str
Product                    str
Quantity                 int64
Unit_Price             float64
Discount_Pct             int64
Sales                  float64
Payment_Mode               str
dtype: object
DataFrame missing values:
 Order_ID        0
Customer_ID     0
Order_Date      0
Region          0
Product         0
Quantity        0
Unit_Price      0
Discount_Pct    0
Sales           0
Payment_Mode    0
dtype: int64


In [23]:
df.duplicated().sum()

np.int64(0)

In [25]:
analysis_date = df["Order_Date"].max() + pd.Timedelta(days=1)
print("Analysis Date:", analysis_date)

Analysis Date: 2026-09-16 00:00:00


In [27]:
# calculate recency 
recency = df.groupby('Customer_ID')['Order_Date'].max().reset_index()
recency.head()

,Customer_ID,Order_Date
0,1001,2026-07-29
1,1002,2026-04-19
2,1003,2026-04-07
3,1004,2026-03-19
4,1005,2026-07-14


In [30]:
# Rename the column:

recency.columns = ['Customer_ID', "Last_Purchase_Date"] 
recency.head()

,Customer_ID,Last_Purchase_Date
0,1001,2026-07-29
1,1002,2026-04-19
2,1003,2026-04-07
3,1004,2026-03-19
4,1005,2026-07-14


In [33]:
# calculate number of days since last purchase 

recency['Recency']=(
    analysis_date - recency['Last_Purchase_Date']
).dt.days
recency.head()

,Customer_ID,Last_Purchase_Date,Recency
0,1001,2026-07-29,49
1,1002,2026-04-19,150
2,1003,2026-04-07,162
3,1004,2026-03-19,181
4,1005,2026-07-14,64


In [39]:
# Calculate Frequency 


frequency = (
    df.groupby('Customer_ID')['Order_ID'] 
    .count()
    .reset_index() 
)




# frequency.head()




In [37]:
# Rename the column to 'Recency'

frequency.columns = ['Customer_ID', 'Frequency']
frequency.head()

,Customer_ID,Frequency
0,1001,2
1,1002,5
2,1003,3
3,1004,2
4,1005,2


In [ ]:
# Calculate Monetary Value 

monetary = (
    df.groupby('Customer_ID')['Sales'].sum().reset_index()
)

# rename the column to monetary 

frequency.columns = ['Customer_ID', 'Monetary'] 
frequency.head()

* Combine R F M 

In [ ]:
rfm = recency.merge(frequency, on ='Customer_ID') 
rfm = rfm.merge(monetary, on ='Customer_ID')

# view the first 10 rows of the RFM table

rfm.head(10)

In [ ]:
""" Very recnet -> 5
Recent -> 4
Average -> 3
Older -> 2
Very Old -> 1 """

rfm['R_Score'] = pd.qcut(
    rfm["Recency"],
    5,
    labels = [1,2,3,4,5]
)


In [ ]:
rfm['F_Score'] = pd.qcut(
    rfm["Frequency"].rank(method = 
    "first"),
    5,
    labels = [1,2,3,4,5] 
)


In [ ]:
rfm['M_Score'] = pd.qcut(
    rfm["Monetary"].rank(method = "first"),
    5,
    labels = [1,2,3,4,5]
)
# higher spending gets a higher score 

In [ ]:
# convert score to number 

rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)
rfm['M_Score'] = rfm['M_Score'].astype(int)


In [ ]:
# combine scores into a single RFM_Score column

rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) + 
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str) 

)

In [ ]:
def segment_customers(row):
    r= row['R_score']
    f= row['F_score']
    m= row['M_score']

if r >= 4 and f >= 4 and m >= 4:
    return "Champion"
elif r >= 3 and f >= 4:
    return "Loyal Customer"
elif r>=4 and f>=2:
    return "Potential Loyalist"
elif r<=2 and f>=3:
    return "At Risk"
else:
    return "Lost Customer"